# BÀI TẬP: MEDICAL INSURANCE COST
**Nguồn:** kaggle.com/datasets/mosapabdelghany/medical-insurance-cost-dataset



## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')


csv_path = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
df.shape
df.columns.tolist()
df.dtypes

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

## A.2. Missing values & Duplicate data

In [3]:
df.isna().sum()
df.duplicated().sum()

np.int64(1)

## A.3. Invalid values

In [4]:
df[(df['age'] < 0) | (df['bmi'] <= 0) | (df['children'] < 0) | (df['charges'] <= 0)]

,age,sex,bmi,children,smoker,region,charges


## A.4. Create a new column
Tạo cột `bmi_group`: Normal (<25), Overweight (25-30), Obese (>=30).

In [7]:
df['bmi_group'] = pd.cut(df['bmi'], bins=[-np.inf, 25, 30, np.inf], labels=['Normal', 'Overweight', 'Obese'], right=False)

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [8]:
numeric_df = df.select_dtypes(include=np.number)
pd.DataFrame({'mean': numeric_df.mean(), 'median': numeric_df.median(), 'mode': numeric_df.mode().iloc[0]})

,mean,median,mode
age,39.207025,39.000,18.0000
bmi,30.663397,30.400,32.3000
children,1.094918,1.000,0.0000
charges,13270.422265,9382.033,1639.5631


## Group 2 — Dispersion

In [9]:
pd.DataFrame({'range': numeric_df.max() - numeric_df.min(), 'variance': numeric_df.var(), 'std': numeric_df.std(), 'IQR': numeric_df.quantile(0.75) - numeric_df.quantile(0.25)})

,range,variance,std,IQR
age,46.00000,1.974014e+02,14.049960,24.000000
bmi,37.17000,3.718788e+01,6.098187,8.397500
children,5.00000,1.453213e+00,1.205493,2.000000
charges,62648.55411,1.466524e+08,12110.011237,11899.625365


## Group 3 — Location and Shape

In [10]:
pd.DataFrame({'Q1': numeric_df.quantile(0.25), 'median': numeric_df.median(), 'Q3': numeric_df.quantile(0.75), 'skewness': numeric_df.skew(), 'kurtosis': numeric_df.kurt()})

,Q1,median,Q3,skewness,kurtosis
age,27.00000,39.000,51.000000,0.055673,-1.245088
bmi,26.29625,30.400,34.693750,0.284047,-0.050732
children,0.00000,1.000,2.000000,0.938380,0.202454
charges,4740.28715,9382.033,16639.912515,1.515880,1.606299


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Người hút thuốc trả chi phí cao hơn bao nhiêu lần so với người không hút, và có đồng đều giữa các vùng không?

In [11]:
charges = df.groupby(['region', 'smoker'])['charges'].mean().unstack()
charges['ratio'] = charges['yes'] / charges['no']
charges.loc['Overall', 'no'] = df.loc[df['smoker'] == 'no', 'charges'].mean()
charges.loc['Overall', 'yes'] = df.loc[df['smoker'] == 'yes', 'charges'].mean()
charges.loc['Overall', 'ratio'] = charges.loc['Overall', 'yes'] / charges.loc['Overall', 'no']
charges

smoker,no,yes,ratio
region,,,
northeast,9165.531672,29673.536473,3.237514
northwest,8556.463715,30192.003182,3.528561
southeast,8032.216309,34844.996824,4.338155
southwest,8019.284513,32269.063494,4.023933
Overall,8434.268298,32050.231832,3.800001


## Câu hỏi 2: BMI có tương quan với chi phí mạnh hơn ở nhóm hút thuốc hay không hút thuốc?

In [13]:
bmi_charge_corr = pd.Series({smoker: group['bmi'].corr(group['charges']) for smoker, group in df.groupby('smoker')}, name='correlation')
bmi_charge_corr

no     0.084037
yes    0.806481
Name: correlation, dtype: float64

## Câu hỏi 3: Vùng nào có chi phí bảo hiểm trung bình cao nhất?

In [14]:
df.groupby('region')['charges'].mean().sort_values(ascending=False)

region
southeast    14735.411438
northeast    13406.384516
northwest    12417.575374
southwest    12346.937377
Name: charges, dtype: float64

## Câu hỏi 4: Số lượng con cái có làm tăng chi phí bảo hiểm không?

In [15]:
children_charges = df.groupby('children')['charges'].agg(['mean', 'count'])
children_charges, df['children'].corr(df['charges'])

(                  mean  count
 children                     
 0         12365.975602    574
 1         12731.171832    324
 2         15073.563734    240
 3         15355.318367    157
 4         13850.656311     25
 5          8786.035247     18,
 np.float64(0.0679982268479048))

## Câu hỏi 5: Tuổi có tương quan với chi phí bảo hiểm không?

In [16]:
stats.pearsonr(df['age'], df['charges'])

PearsonRResult(statistic=np.float64(0.2990081933306479), pvalue=np.float64(4.8866933317182684e-29))

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Chi phí bảo hiểm của người hút thuốc cao khoảng 3,8 lần so với người không hút thuốc, và tỷ lệ này thay đổi giữa các vùng. Khu vực Southeast có chi phí bảo hiểm trung bình cao nhất, trong khi Southwest có mức thấp nhất. BMI có tương quan mạnh với chi phí ở nhóm hút thuốc nhưng tương quan rất yếu ở nhóm không hút thuốc. Số lượng con có tương quan dương rất yếu với chi phí, nên chưa đủ cơ sở kết luận rằng có nhiều con sẽ làm chi phí tăng đều. Phân phối charges lệch phải, cho thấy một số người có chi phí bảo hiểm rất cao so với phần lớn dữ liệu.